# Banking77 Intent Classification with BERT

**Section 2 — Project Framework, Data Loading & Exploratory Analysis (EDA only)**

Scope of this section: CRISP-DM framing, dataset verification, and EDA/visualization.
No model training here — no MLP baseline, no RoBERTa/BERT fine-tuning (those belong to later sections).

## 1. Project Framework — CRISP-DM applied to Banking77

Dataset: Banking77 — 10,003 train / 3,080 test rows, columns `text, category`, 77 customer-service intents.

| # | CRISP-DM phase | How it applies to this project |
|---|---|---|
| 1 | **Business Understanding** | Automate customer-query routing for a bank's support chatbot. Success = high **first-response accuracy** (correct intent on first try) and reduced **triage time** (faster handoff to the right team/flow). Baseline expectation: a majority-class guess would score ~1/77, so any useful model must clearly beat that. |
| 2 | **Data Understanding** | This section (Section 2): verify row counts, schema, 77 intents in each split, missing/duplicates, intent distribution, text-length patterns, and example queries. |
| 3 | **Data Preparation** | Later section: text cleaning (light — keep intent signal), stratified train/validation split, tokenizer encoding, padding/truncation, class-imbalance handling if needed. |
| 4 | **Modeling** | Later sections: classical/MLP baseline first, then fine-tuned transformer (BERT/RoBERTa) for intent classification. |
| 5 | **Evaluation** | Later section: accuracy + macro-F1 (77 classes, imbalance-sensitive), per-intent precision/recall, confusion-matrix review of confusable intents. |
| 6 | **Deployment** | Later section: export model + tokenizer, inference pipeline with confidence threshold and fallback-to-human, monitoring for intent drift. **PII note:** this is a regulated industry (banking) — raw queries may contain emails, phones, or card numbers, so the pipeline must mask/hash PII before logging or training (see PII stub at the end of this notebook). |

## 2. Load and Verify Data

In [ ]:
import pandas as pd
import numpy as np

TRAIN_PATH = "datasets/banking77_train.csv"
TEST_PATH = "datasets/banking77_test.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

# --- Shape and schema assertions ---
assert train_df.shape[0] == 10003, f"Unexpected train rows: {train_df.shape[0]}"
assert test_df.shape[0] == 3080, f"Unexpected test rows: {test_df.shape[0]}"
assert list(train_df.columns) == ["text", "category"], train_df.columns.tolist()
assert list(test_df.columns) == ["text", "category"], test_df.columns.tolist()
assert train_df["category"].nunique() == 77, train_df["category"].nunique()
assert test_df["category"].nunique() == 77, test_df["category"].nunique()

print(f"train: {train_df.shape} | test: {test_df.shape}")
print(f"train intents: {train_df['category'].nunique()} | test intents: {test_df['category'].nunique()}")
print(f"intent sets identical: {set(train_df['category'].unique()) == set(test_df['category'].unique())}")

# --- Data quality: missing values and exact duplicates ---
print("\nMissing values (train):", train_df.isnull().sum().to_dict())
print("Missing values (test):", test_df.isnull().sum().to_dict())
print(f"Exact-duplicate rows — train: {train_df.duplicated().sum()}, test: {test_df.duplicated().sum()}")

# --- Example rows ---
print("\n--- train head ---")
display(train_df.head(5))
print("--- test head ---")
display(test_df.head(5))

## 3. Explore and Visualize

Goals: intent distribution (which intents dominate?), text-length patterns overall and per intent,
representative examples, imbalance assessment, and a PII-handling stub for the regulated banking context.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

intent_counts = train_df["category"].value_counts()
print(f"Intents: {len(intent_counts)} | min count: {intent_counts.min()} | max count: {intent_counts.max()}")
print(f"Mean per intent: {intent_counts.mean():.1f} | std: {intent_counts.std():.1f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top10 = intent_counts.head(10)
sns.barplot(x=top10.values, y=top10.index, ax=axes[0], color="steelblue")
axes[0].set_title("Top 10 intents (train)")
axes[0].set_xlabel("Count")

bottom10 = intent_counts.tail(10).sort_values(ascending=True)
sns.barplot(x=bottom10.values, y=bottom10.index, ax=axes[1], color="coral")
axes[1].set_title("Bottom 10 intents (train)")
axes[1].set_xlabel("Count")

plt.tight_layout()
plt.show()

print("\nTop 10:", top10.to_dict())
print("Bottom 10:", bottom10.sort_values().to_dict())

In [ ]:
# Text-length features (characters and words)
train_df["n_chars"] = train_df["text"].str.len()
train_df["n_words"] = train_df["text"].str.split().str.len()

print("--- Overall text-length stats (train) ---")
display(train_df[["n_chars", "n_words"]].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(train_df["n_chars"], bins=40, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Distribution of query length (characters)")
axes[0].set_xlabel("Characters")

sns.boxplot(x=train_df["n_words"], ax=axes[1], color="lightblue")
axes[1].set_title("Spread of query length (words)")
axes[1].set_xlabel("Words")
plt.tight_layout()
plt.show()

# Per-intent length: which intents run long vs short?
per_intent_len = train_df.groupby("category")["n_words"].agg(["mean", "median", "count"]).round(2)
per_intent_len = per_intent_len.sort_values("mean", ascending=False)
print("--- Longest 5 intents by mean word count ---")
display(per_intent_len.head(5))
print("--- Shortest 5 intents by mean word count ---")
display(per_intent_len.tail(5))

# Boxplot for a readable subset (top-5 most frequent intents)
top5_intents = intent_counts.head(5).index.tolist()
subset = train_df[train_df["category"].isin(top5_intents)]
plt.figure(figsize=(12, 4))
sns.boxplot(data=subset, x="n_words", y="category", order=top5_intents)
plt.title("Word-count spread for top-5 intents")
plt.xlabel("Words")
plt.tight_layout()
plt.show()

In [ ]:
# Representative examples: 2-3 queries for a few sample intents
np.random.seed(42)
sample_intents = ["card_arrival", "lost_or_stolen_card", "exchange_rate", "fiat_currency_support"]
sample_intents = [c for c in sample_intents if c in train_df["category"].unique()]

for intent in sample_intents:
    print(f"=== {intent} ===")
    for q in train_df.loc[train_df['category'] == intent, 'text'].sample(3, random_state=42):
        print(f"  - {q}")
    print()

# --- Class imbalance note ---
ratio = intent_counts.max() / intent_counts.min()
print(f"Imbalance ratio (max/min per-intent count): {ratio:.2f}")
print(
    "Note: Banking77 is roughly balanced by design (each intent has 100+ train examples), "
    "so no resampling is applied in EDA. If the confusion matrix (later section) shows weak recall "
    "on rare/confusable intents, consider stratified splits, class weights, or focal loss during modeling."
)

### PII protection strategy (stub — no real data mutation in EDA)

Banking queries can contain emails, phone numbers, or card numbers. Before any logging,
training export, or deployment demo, PII must be masked or hashed. The stub below defines
detection patterns and a hashing helper, demonstrated on **synthetic strings only** —
the real `train_df`/`test_df` are left untouched in this section.

In [ ]:
import re
import hashlib

# Detection patterns (stub — extend with IBAN/SSN rules per compliance review)
PII_PATTERNS = {
    "email": re.compile(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+", re.IGNORECASE),
    "phone": re.compile(r"\+?\d[\d\s().-]{7,}\d"),
    "card": re.compile(r"\b(?:\d[ -]?){13,19}\b"),  # 13-19 digit sequences (candidate card numbers)
}

def hash_pii(value: str, salt: str = "banking77") -> str:
    """One-way hash for a detected PII value (keeps joinability without storing raw PII)."""
    return hashlib.sha256(f"{salt}:{value}".encode()).hexdigest()[:16]

def find_pii(text: str) -> dict:
    """Return detected PII spans per category (detection only — no mutation)."""
    return {name: pat.findall(text) for name, pat in PII_PATTERNS.items()}

# Demo on synthetic strings only — never on real dataset rows in EDA
demo_texts = [
    "Contact me at jane.doe@example.com about my card",
    "Call me back on +1 555-123-4567 regarding the transfer",
    "My card 4111 1111 1111 1111 is not working",
]
for t in demo_texts:
    print(f"TEXT: {t}")
    print(f"  detected: {find_pii(t)}")
    print(f"  hashed email demo: {hash_pii('jane.doe@example.com')}")
    break  # one hashing example is enough; detection shown for all below
for t in demo_texts[1:]:
    print(f"TEXT: {t}")
    print(f"  detected: {find_pii(t)}")

## Section 2 — Wrap-up

- Data verified: 10,003 train / 3,080 test rows, `text, category` schema, 77 intents per split, no missing values or exact duplicates.
- EDA shows the intent distribution shape, query-length patterns, and representative examples above.
- PII handling is stubbed (detect + hash) and will be enforced before any training/logging step.
- Next (later sections): data preparation, MLP baseline, transformer fine-tuning, evaluation, deployment.